<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Sea Surface Height (SWATH) Data from SWOT**

The SWOT Level 2 Nadir Altimeter Geophysical Data Record (GDR) with Waveforms dataset produced by the Surface Water and Ocean Topography (SWOT) mission provides sea surface height, significant wave height and wind speed measurements from the Poseidon-3C nadir altimeter, a Jason-class dual frequency (Ku/C) altimeter. SWOT is a joint mission between NASA and CNES that launched on December 16, 2022. It aims to measure ocean surface topography with unprecedented resolution and accuracy, as well as map inland water bodies globally. The GDR dataset consists of discrete measurements for each half orbit along the ground track with sampling resolutions of approximately 6-km and 300-m at 1Hz and 20Hz, respectively. The data were processed using restituted auxiliary data and the Precise Orbit Ephemeris (POE). The data are available with latency of < 90 days and distributed in netCDF-4 file format. **Source**: [NASA Earthdata](https://www.earthdata.nasa.gov/es/data/catalog/pocloud-swot-l2-nalt-gdr-2.0-2.0?page=6%2C0).


<span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> Valid Earthdata Login (EDL)

<span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data

<span style='color:#ff6666'><font size="5"> **References**

<font size="3"> **SWOT. (2024)**. <i>SWOT Level 2 Nadir Altimeter Geophysical Data Record with Waveforms Version 2.0</i> [Data set]. NASA Physical Oceanography Distributed Active Archive Center. https://doi.org/10.5067/SWOT-NALT-GDR-2.0


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
# import pydap-specific tools
from pydap.client import open_url,get_cmr_urls
from pydap.client import to_netcdf as dap_to_netcdf
import numpy as np

## Search for OPeNDAP URLs

Will need the concept collection ID (ccid), or the DOI of the data of interest at a minimum.

Can pass additional filters to narrow down the search, such as a time range, bounding box, etc.


In [ ]:
swot_ccid = "C2799438313-POCLOUD"
time_range = [dt.datetime(2023, 1, 1), dt.datetime(2023, 1, 31)] # One month of data

# select a region in the Iceland-Faroe Ridge
bbox = [-20.760731, 60.080727, -4.297294,65.675099] #[west, south, east, north]

cmr_urls = get_cmr_urls(ccid=swot_ccid, bounding_box = bbox, time_range=time_range, limit=1000) # you can incread the limit of results
print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:6]

In [ ]:
# We further filter the return CMR URLs to select the STANDARD dataset, with identifier : SWOT_GPN
swot_standard_urls = [url for url in cmr_urls if url.split('/')[-1][:8]=='SWOT_GPN']
len(swot_standard_urls)

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**does not work with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **2)** above

In [ ]:
%%time
pyds = open_url(swot_standard_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
output_path = "data/"

### Download only coordinate data

<font size="3"> This step will help identify the subset of most of data is downloaded within our bounding box

In [ ]:
%%time
dap_to_netcdf(swot_standard_urls, 
              session=my_session, 
              keep_variables= [
                "/data_01/time", "/data_01/longitude",
                "/data_20/time","/data_20/longitude",
                ],
              output_path=output_path)

## Inspect all downloaded files

<font size="3"> Here, we further identify the subset of data needed on the remote file, that will return ONLY data within our bounding box, for any possible variable of interest.

In [ ]:
## get min lat/lon from 
minLon, maxLon = bbox[0], bbox[2]
slices = []
for url in swot_standard_urls:
    filename = output_path+f"{url.split('/')[-1]}.nc4"
    dt1 = xr.open_datatree(filename).load()
    # find index /data_01/longitude
    longitude = dt1['data_01/longitude']
    longitude = longitude.where(longitude <=180,  longitude -360)
    mask = (longitude >= minLon) & (longitude <= maxLon)
    idx = np.nonzero(mask.values)[0]
    # find index /data_20/longitude
    longitude = dt1['data_20/longitude']
    longitude = longitude.where(longitude <=180,  longitude -360)
    mask = (longitude >= minLon) & (longitude <= maxLon)
    idx2 = np.nonzero(mask.values)[0]
    # create slice for each granule
    slices.append({"/data_01/time":(idx[0],idx[-1]), 
                   "/data_20/time": (idx2[0],idx2[-1])})

## Declare All Variables of interest 

<font size="3"> Using their Fully Qualifying Name

In [ ]:
Variables = [
    "/data_01/time", "/data_01/longitude", "/data_01/latitude", 
    "/data_01/sst", "/data_01/mean_dynamic_topography_interp_qual",
    "/data_01/surface_classification_flag", "/data_01/ice_flag",
    "/data_01/mean_dynamic_topography", "/data_01/rain_flag",
    "/data_20/time", "/data_20/longitude", "/data_20/latitude", 
    "/data_20/distance_to_coast",
]


## Download step

<font size="3"> Finally we download all data of interest, minimizing the data outside of our bounding box


In [ ]:
%%time
dap_to_netcdf(swot_standard_urls, 
              session=my_session, 
              keep_variables=Variables, 
              dim_slices = slices, 
              output_path=output_path)

## Inspect a file

In [ ]:
filename = output_path+f"{swot_standard_urls[0].split('/')[-1]}.nc4"
dt1 = xr.open_datatree(filename).load()
dt1